# 약관 비교 결과(terms-verification) 디렉터리 일괄 삭제

Azure Files의 `terms-verification/` 아래에 rqtKey별 디렉터리가 너무 많이 쌓여서, 포털/Storage Browser에서 하나씩 지우기 어려울 때 쓰는 일괄 삭제 스크립트입니다.

Azure Files 디렉터리는 **비어있어야만 삭제**할 수 있어서(재귀 삭제 API가 따로 없음), 각 rqtKey 디렉터리 안의 파일(`result.json`, 리포트 `.xlsx`)을 먼저 지운 뒤 디렉터리 자체를 지우는 방식으로 동작합니다.

**삭제는 되돌릴 수 없습니다.** 아래 순서로 실행하세요.
1. 목록 조회 셀로 대상 rqtKey 개수를 확인
2. dry-run 셀로 "몇 개가 지워질지"만 미리 확인 (실제 삭제 없음)
3. `TEST_LIMIT`을 작은 값(예: 3)으로 두고 한 번 실행해서 정상 동작하는지 확인
4. 문제없으면 `TEST_LIMIT = None`, `CONFIRM_DELETE = True`로 바꿔서 전체 삭제

## 사전 준비
```bash
pip install azure-storage-file-share python-dotenv
```
프로젝트 루트(`azure-doc-ai-service/`)의 `.env`에 아래 값이 채워져 있어야 합니다.
```
AZURE_FILE_STORAGE_CONNECTION_STRING=<연결 문자열>
AZURE_FILE_SHARE_NAME=documents
```

In [ ]:
import os
from pathlib import Path

from azure.core.exceptions import ResourceNotFoundError
from azure.storage.fileshare import ShareServiceClient
from dotenv import load_dotenv

# 이 노트북(azure-doc-ai-service/notebooks/)의 부모 폴더(azure-doc-ai-service/)에 있는 .env를 로드
ENV_PATH = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=ENV_PATH)

CONNECTION_STRING = os.environ["AZURE_FILE_STORAGE_CONNECTION_STRING"]
SHARE_NAME = os.environ.get("AZURE_FILE_SHARE_NAME", "documents")

# app/services/terms_verification_service.py의 RESULT_ROOT와 동일한 값
ROOT_DIR = "terms-verification"

service_client = ShareServiceClient.from_connection_string(CONNECTION_STRING)
share_client = service_client.get_share_client(SHARE_NAME)
root_client = share_client.get_directory_client(ROOT_DIR)

print("SHARE_NAME:", SHARE_NAME)
print("ROOT_DIR:", ROOT_DIR)

## 1. 삭제 대상(rqtKey 디렉터리) 목록 조회

`ROOT_DIR` 바로 아래에 있는 디렉터리만 뽑습니다 (rqtKey 하나당 디렉터리 하나).

In [ ]:
rqt_key_dirs = [item.name for item in root_client.list_directories_and_files() if item.is_directory]

print(f"'{ROOT_DIR}' 아래 rqtKey 디렉터리 수: {len(rqt_key_dirs)}개")
print("\n앞부분 미리보기 (최대 10개):")
for name in rqt_key_dirs[:10]:
    print(f"  - {name}")

## 2. Dry-run - 실제로 몇 개가 지워질지 미리 확인

디렉터리 안의 파일 목록만 조회하고, **아직 아무것도 지우지 않습니다.**

In [ ]:
def count_files(rqt_key: str) -> int:
    directory_client = share_client.get_directory_client(f"{ROOT_DIR}/{rqt_key}")
    return sum(1 for item in directory_client.list_directories_and_files())


sample = rqt_key_dirs[:20]  # 전체를 다 세면 느릴 수 있어서 앞부분만 표본으로 확인
sample_file_counts = {rqt_key: count_files(rqt_key) for rqt_key in sample}

print(f"표본 {len(sample)}개 디렉터리의 파일 개수:")
for rqt_key, count in sample_file_counts.items():
    print(f"  - {rqt_key}: 파일 {count}개")

print(f"\n실제 삭제를 실행하면 rqtKey 디렉터리 {len(rqt_key_dirs)}개(와 그 안의 파일 전부)가 지워집니다.")

## 3. 실제 삭제 실행 (되돌릴 수 없음)

`TEST_LIMIT`을 작은 값으로 먼저 실행해서 정상 동작을 확인한 뒤, 전체를 지우려면 `TEST_LIMIT = None`으로 바꾸세요.
두 조건(`TEST_LIMIT`, `CONFIRM_DELETE`)을 실수로 한 번에 True로 두지 않도록 기본값은 안전한 쪽(작은 개수 + 미실행)으로 되어 있습니다.

In [ ]:
TEST_LIMIT = 3       # None이면 전체 삭제. 먼저 작은 값으로 확인한 뒤 None으로 바꾸세요.
CONFIRM_DELETE = False  # 실제로 지우려면 True로 바꾼 뒤 이 셀을 다시 실행하세요.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

MAX_WORKERS = 8


def delete_directory_recursive(directory_client) -> tuple[int, int]:
    """directory_client가 가리키는 디렉터리 안을 전부 비우고, 마지막에 그 디렉터리 자체도 삭제한다.
    Azure Files 디렉터리는 비어있어야 삭제할 수 있어서, 하위 파일/디렉터리부터 먼저 지워야 한다."""
    file_count = 0
    dir_count = 0
    for item in list(directory_client.list_directories_and_files()):
        if item.is_directory:
            sub_files, sub_dirs = delete_directory_recursive(directory_client.get_subdirectory_client(item.name))
            file_count += sub_files
            dir_count += sub_dirs
        else:
            directory_client.delete_file(item.name)
            file_count += 1
    directory_client.delete_directory()
    dir_count += 1
    return file_count, dir_count


def delete_rqt_key_dir(rqt_key: str) -> tuple[int, int]:
    directory_client = share_client.get_directory_client(f"{ROOT_DIR}/{rqt_key}")
    try:
        return delete_directory_recursive(directory_client)
    except ResourceNotFoundError:
        return 0, 0


targets = rqt_key_dirs if TEST_LIMIT is None else rqt_key_dirs[:TEST_LIMIT]

if not CONFIRM_DELETE:
    print(f"CONFIRM_DELETE가 False라 삭제를 실행하지 않았습니다. (대상 {len(targets)}개 준비됨)")
    print("위 dry-run 결과를 확인한 뒤 CONFIRM_DELETE = True로 바꿔서 이 셀을 다시 실행하세요.")
else:
    total_files = 0
    total_dirs = 0
    failed: list[tuple[str, str]] = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(delete_rqt_key_dir, rqt_key): rqt_key for rqt_key in targets}
        for i, future in enumerate(as_completed(futures), start=1):
            rqt_key = futures[future]
            try:
                files, dirs = future.result()
                total_files += files
                total_dirs += dirs
            except Exception as exc:
                failed.append((rqt_key, str(exc)))
            if i % 50 == 0 or i == len(targets):
                print(f"진행: {i}/{len(targets)}")

    print(f"\n디렉터리 {total_dirs}개, 파일 {total_files}개 삭제 완료 (대상 {len(targets)}개 중 실패 {len(failed)}건)")
    for rqt_key, err in failed:
        print(f"  실패 - {rqt_key}: {err}")

## 4. 삭제 결과 확인

`ROOT_DIR` 아래에 남은 rqtKey 디렉터리 수를 다시 조회합니다. `TEST_LIMIT`을 걸고 실행했다면 그만큼만 줄어든 게 정상입니다.

In [ ]:
remaining = [item.name for item in root_client.list_directories_and_files() if item.is_directory]
print(f"'{ROOT_DIR}' 아래 남은 rqtKey 디렉터리 수: {len(remaining)}개")